<a href="https://colab.research.google.com/github/Langsch/BD3-Projetos/blob/main/ROLAP_Postgresql_BD3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update -y
!apt-get install postgresql postgresql-contrib -y
!service postgresql start
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';"
!sudo -u postgres createdb bd3 || true
!service postgresql restart
!sleep 5

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Fetched 3,917 B in 1s (3,393 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
postgresql is already the newest version (14+238).

In [ ]:
script_rio_sql = """
DROP SCHEMA IF EXISTS imobiliaria_rio_de_janeiro CASCADE;
CREATE SCHEMA imobiliaria_rio_de_janeiro;

-- Criação da tabela
-- Pessoa
CREATE TABLE imobiliaria_rio_de_janeiro.pessoa (
    id_pessoa  SERIAL        PRIMARY KEY,
    cpf        VARCHAR(11)   NOT NULL,
    tel_ddd    CHAR(2),
    tel_nro    VARCHAR(9),
    email      VARCHAR(100),
    nome       VARCHAR(100)  NOT NULL
);

-- Corretor
CREATE TABLE imobiliaria_rio_de_janeiro.corretor (
    id_pessoa  INT           PRIMARY KEY,
    salario    DECIMAL(10,2),
    CONSTRAINT fk_corretor_pessoa
        FOREIGN KEY (id_pessoa)
        REFERENCES imobiliaria_rio_de_janeiro.pessoa(id_pessoa)
        ON DELETE CASCADE
        ON UPDATE RESTRICT
);

-- Usuario
CREATE TABLE imobiliaria_rio_de_janeiro.usuario (
    id_pessoa  INT           PRIMARY KEY,
    cnd        VARCHAR(200),
    CONSTRAINT fk_usuario_pessoa
        FOREIGN KEY (id_pessoa)
        REFERENCES imobiliaria_rio_de_janeiro.pessoa(id_pessoa)
        ON DELETE CASCADE
        ON UPDATE RESTRICT
);

-- Imovel
CREATE TABLE imobiliaria_rio_de_janeiro.imovel (
   id_imovel   SERIAL        PRIMARY KEY,
    tipo       VARCHAR(50),
    valor      DECIMAL(12,2),
    area       DECIMAL(10,2),
    estado     VARCHAR(50),
    cidade     VARCHAR(50),
    bairro     VARCHAR(50),
    rua        VARCHAR(50),
    numero     VARCHAR(10),
    complemento VARCHAR(50)
);

-- DonoImovel
CREATE TABLE imobiliaria_rio_de_janeiro.dono_imovel (
   id_imovel   INT           NOT NULL,
    id_pessoa  INT           NOT NULL,
    PRIMARY KEY (id_imovel, id_pessoa),
    CONSTRAINT fk_dono_imovel
        FOREIGN KEY (id_imovel)
        REFERENCES imobiliaria_rio_de_janeiro.imovel(id_imovel)
        ON DELETE CASCADE
        ON UPDATE RESTRICT,
    CONSTRAINT fk_dono_usuario
        FOREIGN KEY (id_pessoa)
        REFERENCES imobiliaria_rio_de_janeiro.usuario(id_pessoa)
        ON DELETE CASCADE
        ON UPDATE RESTRICT
);

-- VendedorResponsavel
CREATE TABLE imobiliaria_rio_de_janeiro.vendedor_responsavel (
   id_imovel     INT         NOT NULL,
    id_corretor  INT         NOT NULL,
    PRIMARY KEY (id_imovel, id_corretor),
    CONSTRAINT fk_vr_imovel
        FOREIGN KEY (id_imovel)
        REFERENCES imobiliaria_rio_de_janeiro.imovel(id_imovel)
        ON DELETE CASCADE
        ON UPDATE RESTRICT,
    CONSTRAINT fk_vr_corretor
        FOREIGN KEY (id_corretor)
        REFERENCES imobiliaria_rio_de_janeiro.corretor(id_pessoa)
        ON DELETE CASCADE
        ON UPDATE RESTRICT
);

-- Negociacao (histórico, protegido contra delete acidental)
CREATE TABLE imobiliaria_rio_de_janeiro.negociacao (
    id_cliente  INT          NOT NULL,
   id_imovel    INT          NOT NULL,
    status      VARCHAR(30),
    data_ini    DATE,
    data_fim    DATE,
    tipo        VARCHAR(30),
    PRIMARY KEY (id_cliente, id_imovel),
    CONSTRAINT fk_neg_usuario
        FOREIGN KEY (id_cliente)
        REFERENCES imobiliaria_rio_de_janeiro.usuario(id_pessoa)
        ON DELETE RESTRICT
        ON UPDATE RESTRICT,
    CONSTRAINT fk_neg_imovel
        FOREIGN KEY (id_imovel)
        REFERENCES imobiliaria_rio_de_janeiro.imovel(id_imovel)
        ON DELETE RESTRICT
        ON UPDATE RESTRICT
);

-- ImagemImovel
CREATE TABLE imobiliaria_rio_de_janeiro.imagem_imovel (
   id_imovel   INT           NOT NULL,
    path_img   VARCHAR(300)  NOT NULL,
    PRIMARY KEY (id_imovel, path_img),
    CONSTRAINT fk_img_imovel
        FOREIGN KEY (id_imovel)
        REFERENCES imobiliaria_rio_de_janeiro.imovel(id_imovel)
        ON DELETE CASCADE
        ON UPDATE RESTRICT
);
"""
with open("script_rio.sql", "w") as f:
    f.write(script_rio_sql)

!psql -U postgres -h localhost -d bd3 -f script_rio.sql

Password for user postgres: 
psql:script_rio.sql:2: NOTICE:  drop cascades to 8 other objects
DETAIL:  drop cascades to table imobiliaria_rio_de_janeiro.pessoa
drop cascades to table imobiliaria_rio_de_janeiro.corretor
drop cascades to table imobiliaria_rio_de_janeiro.usuario
drop cascades to table imobiliaria_rio_de_janeiro.imovel
drop cascades to table imobiliaria_rio_de_janeiro.dono_imovel
drop cascades to table imobiliaria_rio_de_janeiro.vendedor_responsavel
drop cascades to table imobiliaria_rio_de_janeiro.negociacao
drop cascades to table imobiliaria_rio_de_janeiro.imagem_imovel
DROP SCHEMA
CREATE SCHEMA
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE


In [ ]:
import psycopg2
import random
from datetime import date, timedelta

# Connection details
conn_details = {
    "dbname": "bd3",
    "user": "postgres",
    "password": "postgres",
    "host": "localhost"
}

conn = None
cur = None

try:
    conn = psycopg2.connect(**conn_details)
    cur = conn.cursor()

    # --- Data Generation --- START

    # Helper function to generate common person data
    def generate_person_details(num_people):
        people_data = []
        for i in range(num_people):
            cpf = ''.join([str(random.randint(0, 9)) for _ in range(11)])
            tel_ddd = str(random.randint(11, 99))
            tel_nro = ''.join([str(random.randint(0, 9)) for _ in range(9)])
            nome = f"Pessoa {i+1} Teste"
            email = f"pessoa{i+1}@example.com"
            people_data.append((cpf, tel_ddd, tel_nro, email, nome))
        return people_data

    # Helper function to generate imovel data
    def generate_imovel_details(num_imoveis, city):
        imovel_types = ["Casa", "Apartamento", "Terreno", "Comercial"]
        estados = ["RJ", "SP", "MG"]
        bairros = ["Icarai", "Centro", "Fonseca", "Piratininga", "Copacabana", "Ipanema"]
        imoveis_data = []
        for i in range(num_imoveis):
            tipo = random.choice(imovel_types)
            valor = round(random.uniform(100000.00, 5000000.00), 2)
            area = round(random.uniform(50.00, 1000.00), 2)
            estado = "RJ" # Assuming RJ for both cities
            cidade = city
            bairro = random.choice(bairros)
            rua = f"Rua {random.randint(1, 200)}"
            numero = str(random.randint(1, 1000))
            complemento = random.choice([f"Apto {random.randint(101, 500)}", f"Casa {random.randint(1, 50)}", None])
            imoveis_data.append((tipo, valor, area, estado, cidade, bairro, rua, numero, complemento))
        return imoveis_data

    num_people_to_generate = 10
    num_imoveis_to_generate = 10

    # Generate base person details (similar customers)
    base_person_details = generate_person_details(num_people_to_generate)

    # --- imobiliaria_niteroi ---
    print("\n--- Generating data for imobiliaria_niteroi ---")
    niteroi_pessoa_ids = []
    insert_pessoa_sql_niteroi = "INSERT INTO imobiliaria_niteroi.pessoa (cpf, tel_ddd, tel_nro, email, nome) VALUES (%s, %s, %s, %s, %s) RETURNING id_pessoa;"
    for data in base_person_details:
        cur.execute(insert_pessoa_sql_niteroi, data)
        niteroi_pessoa_ids.append(cur.fetchone()[0])
    print(f"Inserted {len(niteroi_pessoa_ids)} pessoas into imobiliaria_niteroi.")

    random.shuffle(niteroi_pessoa_ids)
    niteroi_corretor_ids = niteroi_pessoa_ids[:3]
    niteroi_usuario_ids = niteroi_pessoa_ids[3:]

    niteroi_corretor_ids_actual = []
    for id_p in niteroi_corretor_ids:
        salario = round(random.uniform(2000.00, 10000.00), 2)
        cur.execute("INSERT INTO imobiliaria_niteroi.corretor (id_pessoa, salario) VALUES (%s, %s);", (id_p, salario))
        niteroi_corretor_ids_actual.append(id_p)
    print(f"Inserted {len(niteroi_corretor_ids_actual)} corretores into imobiliaria_niteroi.")

    niteroi_usuario_ids_actual = []
    for id_p in niteroi_usuario_ids:
        cnd = f"CND-{random.randint(1000, 99999)}"
        cur.execute("INSERT INTO imobiliaria_niteroi.usuario (id_pessoa, cnd) VALUES (%s, %s);", (id_p, cnd))
        niteroi_usuario_ids_actual.append(id_p)
    print(f"Inserted {len(niteroi_usuario_ids_actual)} usuarios into imobiliaria_niteroi.")

    niteroi_imovel_ids = []
    niteroi_imovel_details = generate_imovel_details(num_imoveis_to_generate, "Niteroi")
    for details in niteroi_imovel_details:
        cur.execute("INSERT INTO imobiliaria_niteroi.imovel (tipo, valor, area, estado, cidade, bairro, rua, numero, complemento) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s) RETURNING id_imovel;", details)
        niteroi_imovel_ids.append(cur.fetchone()[0])
    print(f"Inserted {len(niteroi_imovel_ids)} imoveis into imobiliaria_niteroi.")

    # DonoImovel Niteroi
    num_dono_imovel_entries_niteroi = min(len(niteroi_imovel_ids), len(niteroi_usuario_ids_actual), 7)
    for i in range(num_dono_imovel_entries_niteroi):
        id_imovel = random.choice(niteroi_imovel_ids)
        id_pessoa_dono = random.choice(niteroi_usuario_ids_actual)
        cur.execute("INSERT INTO imobiliaria_niteroi.dono_imovel (id_imovel, id_pessoa) VALUES (%s, %s) ON CONFLICT (id_imovel, id_pessoa) DO NOTHING;", (id_imovel, id_pessoa_dono))
    print(f"Inserted dono_imovel relationships for imobiliaria_niteroi.")

    # VendedorResponsavel Niteroi
    num_vendedor_resp_entries_niteroi = min(len(niteroi_imovel_ids), len(niteroi_corretor_ids_actual), 7)
    for i in range(num_vendedor_resp_entries_niteroi):
        id_imovel = random.choice(niteroi_imovel_ids)
        id_corretor_resp = random.choice(niteroi_corretor_ids_actual)
        cur.execute("INSERT INTO imobiliaria_niteroi.vendedor_responsavel (id_imovel, id_corretor) VALUES (%s, %s) ON CONFLICT (id_imovel, id_corretor) DO NOTHING;", (id_imovel, id_corretor_resp))
    print(f"Inserted vendedor_responsavel relationships for imobiliaria_niteroi.")

    # Negociacao Niteroi
    neg_statuses = ["Pendente", "Concluida", "Cancelada"]
    neg_types = ["Venda", "Aluguel"]
    num_negociacao_entries_niteroi = min(len(niteroi_usuario_ids_actual), len(niteroi_imovel_ids), 5)
    for i in range(num_negociacao_entries_niteroi):
        id_cliente = random.choice(niteroi_usuario_ids_actual)
        id_imovel_neg = random.choice(niteroi_imovel_ids)
        status = random.choice(neg_statuses)
        data_ini = date.today() - timedelta(days=random.randint(30, 365))
        data_fim = data_ini + timedelta(days=random.randint(30, 180)) if status == "Concluida" else None
        tipo = random.choice(neg_types)
        cur.execute("INSERT INTO imobiliaria_niteroi.negociacao (id_cliente, id_imovel, status, data_ini, data_fim, tipo) VALUES (%s, %s, %s, %s, %s, %s) ON CONFLICT (id_cliente, id_imovel) DO NOTHING;",
                    (id_cliente, id_imovel_neg, status, data_ini, data_fim, tipo))
    print(f"Inserted negociacao entries for imobiliaria_niteroi.")

    # ImagemImovel Niteroi
    for imovel_id in niteroi_imovel_ids:
        for i in range(random.randint(1, 3)):
            path_img = f"/images/imovel_niteroi_{imovel_id}/photo_{i+1}.jpg"
            cur.execute("INSERT INTO imobiliaria_niteroi.imagem_imovel (id_imovel, path_img) VALUES (%s, %s) ON CONFLICT (id_imovel, path_img) DO NOTHING;", (imovel_id, path_img))
    print(f"Inserted imagem_imovel entries for imobiliaria_niteroi.")

    # --- imobiliaria_rio_de_janeiro ---
    print("\n--- Generating data for imobiliaria_rio_de_janeiro ---")
    rio_pessoa_ids = []
    insert_pessoa_sql_rio = "INSERT INTO imobiliaria_rio_de_janeiro.pessoa (cpf, tel_ddd, tel_nro, email, nome) VALUES (%s, %s, %s, %s, %s) RETURNING id_pessoa;"
    for data in base_person_details:
        cur.execute(insert_pessoa_sql_rio, data)
        rio_pessoa_ids.append(cur.fetchone()[0])
    print(f"Inserted {len(rio_pessoa_ids)} pessoas into imobiliaria_rio_de_janeiro.")

    random.shuffle(rio_pessoa_ids)
    rio_corretor_ids = rio_pessoa_ids[:3]
    rio_usuario_ids = rio_pessoa_ids[3:]

    rio_corretor_ids_actual = []
    for id_p in rio_corretor_ids:
        salario = round(random.uniform(2000.00, 10000.00), 2)
        cur.execute("INSERT INTO imobiliaria_rio_de_janeiro.corretor (id_pessoa, salario) VALUES (%s, %s);", (id_p, salario))
        rio_corretor_ids_actual.append(id_p)
    print(f"Inserted {len(rio_corretor_ids_actual)} corretores into imobiliaria_rio_de_janeiro.")

    rio_usuario_ids_actual = []
    for id_p in rio_usuario_ids:
        cnd = f"CND-{random.randint(1000, 99999)}"
        cur.execute("INSERT INTO imobiliaria_rio_de_janeiro.usuario (id_pessoa, cnd) VALUES (%s, %s);", (id_p, cnd))
        rio_usuario_ids_actual.append(id_p)
    print(f"Inserted {len(rio_usuario_ids_actual)} usuarios into imobiliaria_rio_de_janeiro.")

    rio_imovel_ids = []
    rio_imovel_details = generate_imovel_details(num_imoveis_to_generate, "Rio de Janeiro")
    for details in rio_imovel_details:
        cur.execute("INSERT INTO imobiliaria_rio_de_janeiro.imovel (tipo, valor, area, estado, cidade, bairro, rua, numero, complemento) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s) RETURNING id_imovel;", details)
        rio_imovel_ids.append(cur.fetchone()[0])
    print(f"Inserted {len(rio_imovel_ids)} imoveis into imobiliaria_rio_de_janeiro.")

    # DonoImovel Rio
    num_dono_imovel_entries_rio = min(len(rio_imovel_ids), len(rio_usuario_ids_actual), 7)
    for i in range(num_dono_imovel_entries_rio):
        id_imovel = random.choice(rio_imovel_ids)
        id_pessoa_dono = random.choice(rio_usuario_ids_actual)
        cur.execute("INSERT INTO imobiliaria_rio_de_janeiro.dono_imovel (id_imovel, id_pessoa) VALUES (%s, %s) ON CONFLICT (id_imovel, id_pessoa) DO NOTHING;", (id_imovel, id_pessoa_dono))
    print(f"Inserted dono_imovel relationships for imobiliaria_rio_de_janeiro.")

    # VendedorResponsavel Rio
    num_vendedor_resp_entries_rio = min(len(rio_imovel_ids), len(rio_corretor_ids_actual), 7)
    for i in range(num_vendedor_resp_entries_rio):
        id_imovel = random.choice(rio_imovel_ids)
        id_corretor_resp = random.choice(rio_corretor_ids_actual)
        cur.execute("INSERT INTO imobiliaria_rio_de_janeiro.vendedor_responsavel (id_imovel, id_corretor) VALUES (%s, %s) ON CONFLICT (id_imovel, id_corretor) DO NOTHING;", (id_imovel, id_corretor_resp))
    print(f"Inserted vendedor_responsavel relationships for imobiliaria_rio_de_janeiro.")

    # Negociacao Rio
    num_negociacao_entries_rio = min(len(rio_usuario_ids_actual), len(rio_imovel_ids), 5)
    for i in range(num_negociacao_entries_rio):
        id_cliente = random.choice(rio_usuario_ids_actual)
        id_imovel_neg = random.choice(rio_imovel_ids)
        status = random.choice(neg_statuses)
        data_ini = date.today() - timedelta(days=random.randint(30, 365))
        data_fim = data_ini + timedelta(days=random.randint(30, 180)) if status == "Concluida" else None
        tipo = random.choice(neg_types)
        cur.execute("INSERT INTO imobiliaria_rio_de_janeiro.negociacao (id_cliente, id_imovel, status, data_ini, data_fim, tipo) VALUES (%s, %s, %s, %s, %s, %s) ON CONFLICT (id_cliente, id_imovel) DO NOTHING;",
                    (id_cliente, id_imovel_neg, status, data_ini, data_fim, tipo))
    print(f"Inserted negociacao entries for imobiliaria_rio_de_janeiro.")

    # ImagemImovel Rio
    for imovel_id in rio_imovel_ids:
        for i in range(random.randint(1, 3)):
            path_img = f"/images/imovel_rio_{imovel_id}/photo_{i+1}.jpg"
            cur.execute("INSERT INTO imobiliaria_rio_de_janeiro.imagem_imovel (id_imovel, path_img) VALUES (%s, %s) ON CONFLICT (id_imovel, path_img) DO NOTHING;", (imovel_id, path_img))
    print(f"Inserted imagem_imovel entries for imobiliaria_rio_de_janeiro.")

    # --- Data Generation --- END

    # Commit all changes
    conn.commit()
    print("\nAll data inserted correctly!")

except psycopg2.Error as e:
    print(f"Erro no SGBD: {e}")
    if conn:
        conn.rollback()
except Exception as e:
    print(f"Erro inesperado: {e}")
finally:
    if cur:
        cur.close()
    if conn:
        conn.close()


--- Generating data for imobiliaria_niteroi ---
Inserted 10 pessoas into imobiliaria_niteroi.
Inserted 3 corretores into imobiliaria_niteroi.
Inserted 7 usuarios into imobiliaria_niteroi.
Inserted 10 imoveis into imobiliaria_niteroi.
Inserted dono_imovel relationships for imobiliaria_niteroi.
Inserted vendedor_responsavel relationships for imobiliaria_niteroi.
Inserted negociacao entries for imobiliaria_niteroi.
Inserted imagem_imovel entries for imobiliaria_niteroi.

--- Generating data for imobiliaria_rio_de_janeiro ---
Inserted 10 pessoas into imobiliaria_rio_de_janeiro.
Inserted 3 corretores into imobiliaria_rio_de_janeiro.
Inserted 7 usuarios into imobiliaria_rio_de_janeiro.
Inserted 10 imoveis into imobiliaria_rio_de_janeiro.
Inserted dono_imovel relationships for imobiliaria_rio_de_janeiro.
Inserted vendedor_responsavel relationships for imobiliaria_rio_de_janeiro.
Inserted negociacao entries for imobiliaria_rio_de_janeiro.
Inserted imagem_imovel entries for imobiliaria_rio_de_

In [ ]:
script_sql = """

  DROP SCHEMA IF EXISTS imobiliaria_olap CASCADE;
  CREATE SCHEMA imobiliaria_olap;

  -- Criação da dimensao corretor
  CREATE TABLE imobiliaria_olap.d_corretor (
    id_corretor_dw  INT        PRIMARY KEY,
    nome            VARCHAR(131)  NOT NULL,
    salario_base    FLOAT,
    origem    VARCHAR(32),
    ultima_atualizacao TIMESTAMP
  );

  -- Criação da dimensao endereco
  CREATE TABLE imobiliaria_olap.d_endereco (
    id_endereço_dw  BIGINT        PRIMARY KEY,
    cidade          VARCHAR(127)  NOT NULL,
    bairro          VARCHAR(127)  NOT NULL,
    estado          VARCHAR(127)  NOT NULL,
    origem    VARCHAR(32),
    ultima_atualizacao TIMESTAMP
  );

    -- Criação da dimensao tipo imovel
  CREATE TABLE imobiliaria_olap.d_tipo_imovel (
    id_tipo_imovel_dw  INT        PRIMARY KEY,
    tipo          VARCHAR(31)  NOT NULL,
    origem    VARCHAR(32),
    ultima_atualizacao TIMESTAMP
  );

  -- Criação da dimensao tempo
  CREATE TABLE imobiliaria_olap.d_tempo (
    id_tempo_dw  BIGINT        PRIMARY KEY,
    ano          INT  NOT NULL,
    mes          INT  NOT NULL,
    origem    VARCHAR(32),
    ultima_atualizacao TIMESTAMP
  );

    -- Criação da dimensao indice de ajuste
  CREATE TABLE imobiliaria_olap.d_indice (
    id_indice_dw  INT        PRIMARY KEY,
    nome          VARCHAR(15)  NOT NULL,
    origem    VARCHAR(32),
    ultima_atualizacao TIMESTAMP
  );

  -- Criação da fato venda (analise interna de desempenho de vendedores )
  CREATE TABLE imobiliaria_olap.f_vendas (
    id_corretor  INT,
    id_endereco  BIGINT,
    id_tipo_imovel  INT,
    id_tempo  BIGINT,
    valor_venda  FLOAT,
    tempo_negociacao_dias  INT,
    percentual_comissao FLOAT,
    origem    VARCHAR(32),
    ultima_atualizacao TIMESTAMP,
    PRIMARY KEY (id_corretor, id_endereco, id_tipo_imovel, id_tempo),
    FOREIGN KEY (id_corretor) REFERENCES imobiliaria_olap.d_corretor(id_corretor_dw),
    FOREIGN KEY (id_endereco) REFERENCES imobiliaria_olap.d_endereco(id_endereço_dw),
    FOREIGN KEY (id_tipo_imovel) REFERENCES imobiliaria_olap.d_tipo_imovel(id_tipo_imovel_dw),
    FOREIGN KEY (id_tempo) REFERENCES imobiliaria_olap.d_tempo(id_tempo_dw)
  );



"""

with open("script.sql", "w") as f:
    f.write(script_sql)

!sudo -u postgres psql -d bd3 -f script.sql

psql:script.sql:3: NOTICE:  schema "imobiliaria_olap" does not exist, skipping
DROP SCHEMA
CREATE SCHEMA
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE


In [ ]:
import psycopg2
from datetime import datetime

# Cria a string de conexão
conn_details = {
    "dbname": "bd3",
    "user": "postgres",
    "password": "postgres",
    "host": "localhost"
}

conn = None
cur = None

try:
    conn = psycopg2.connect(**conn_details)
    cur = conn.cursor()

    # 1. Extrai dados de imobiliaria_niteroi.corretor e imobiliaria_niteroi.pessoa
    print("Extraindo dados para d_corretor...")
    select_corretor_sql = """
    SELECT
        c.id_pessoa,
        p.nome,
        c.salario
    FROM
        imobiliaria_niteroi.corretor c
    JOIN
        imobiliaria_niteroi.pessoa p ON c.id_pessoa = p.id_pessoa;
    """
    cur.execute(select_corretor_sql)
    corretor_source_data = cur.fetchall()
    print(f"Extraídos {len(corretor_source_data)} registros da origem.")

    # 2. Transform and Load into imobiliaria_olap.d_corretor
    print("Transformando dados de d_corretor...")
    insert_d_corretor_sql = """
    INSERT INTO imobiliaria_olap.d_corretor (
        id_corretor_dw,
        nome,
        salario_base,
        origem,
        ultima_atualizacao
    ) VALUES (%s, %s, %s, %s, %s)
    ON CONFLICT (id_corretor_dw) DO UPDATE
    SET
        nome = EXCLUDED.nome,
        salario_base = EXCLUDED.salario_base,
        origem = EXCLUDED.origem,
        ultima_atualizacao = EXCLUDED.ultima_atualizacao;
    """

    current_timestamp = datetime.now()
    inserted_count = 0
    for row in corretor_source_data:
        id_pessoa, nome, salario = row
        # Transformação (simples mapeamento)
        id_corretor_dw = id_pessoa
        salario_base = float(salario) # Garantindo que é um float
        origem = 'imobiliaria_niteroi' #indicando origem do dado

        cur.execute(insert_d_corretor_sql, (
            id_corretor_dw,
            nome,
            salario_base,
            origem,
            current_timestamp
        ))
        inserted_count += 1

    # Commit all changes
    conn.commit()
    print(f"Realizado com sucesso a inserção de {inserted_count} tuplas.")
    print("Processo de ETL completo!")

except psycopg2.Error as e:
    print(f"Erro de banco durante ETL para d_corretor: {e}")
    if conn:
        conn.rollback()
except Exception as e:
    print(f"Erro inesperado: {e}")
finally:
    if cur:
        cur.close()
    if conn:
        conn.close()

Extraindo dados para d_corretor...
Extraídos 6 registros da origem.
Transformando dados de d_corretor...
Realizado com sucesso a inserção de 6 tuplas.
Processo de ETL completo!


In [ ]:
!pip install psycopg2-binary

import psycopg2
import pandas as pd

conn = psycopg2.connect(
    dbname="bd3",
    user="postgres",
    password="postgres",
    host="localhost"
)



cur = conn.cursor()
query = "SELECT * FROM imobiliaria_olap.d_corretor;"
cur.execute(query)
df = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
print(df)
df.to_csv("corretores.csv", index=False)
print("CSV gerado com sucesso!")
cur.close()
conn.close()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 21.7 MB/s eta 0:00:00


   id_corretor_dw             nome  salario_base               origem  \
0               9   Pessoa 9 Teste       5598.42  imobiliaria_niteroi   
1              10  Pessoa 10 Teste       3894.29  imobiliaria_niteroi   
2               4   Pessoa 4 Teste       5877.86  imobiliaria_niteroi   
3              14   Pessoa 4 Teste       8008.18  imobiliaria_niteroi   
4              16   Pessoa 6 Teste       3832.22  imobiliaria_niteroi   
5              12   Pessoa 2 Teste       8939.70  imobiliaria_niteroi   

          ultima_atualizacao  
0 2026-04-09 20:58:43.116865  
1 2026-04-09 20:58:43.116865  
2 2026-04-09 20:58:43.116865  
3 2026-04-09 20:58:43.116865  
4 2026-04-09 20:58:43.116865  
5 2026-04-09 20:58:43.116865  
CSV gerado com sucesso!
